# `select_points_in_box()`

The geometry function `nematics3d.geometry.select_points_in_box()` selects three-dimensional points that lie inside or on an oriented rectangular box. The box may be translated and rotated; it does not need to be aligned with the coordinate axes.

The box is defined by one reference corner and three mutually perpendicular outgoing edges. The function can return either the selected points alone or the selected points together with a boolean mask over the original point array. See [Details](#Details) for the local-coordinate test used internally.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** Run the following cell to import `NumPy` and `Nematics3D`; no setup detail is needed to understand the examples.


In [ ]:
import numpy as np
import nematics3d as n3d

## Minimal example: select points inside a box

The following box spans $0 \le x \le 2$, $0 \le y \le 1$, and $0 \le z \le 1$. `get_box_corners()` creates its eight corners in the ordering expected by `select_points_in_box()`.


In [ ]:
corners = n3d.geometry.get_box_corners(2.0, 1.0, 1.0)
points = np.array([
    [0.5, 0.5, 0.5],
    [2.0, 0.5, 0.5],
    [2.2, 0.5, 0.5],
    [-0.1, 0.0, 0.0],
])

selected = n3d.geometry.select_points_in_box(points, corners)
print(selected)

The first point is strictly inside the box and the second lies exactly on the face $x=2$, so both are selected. The remaining two points lie outside. Boundary points are included by design.


## Inputs and outputs

The public signature is:

```python
select_points_in_box(
    points,
    corners,
    is_return_mask=False,
    *,
    atol=1e-9,
)
```

### Accepted point and box representations

`points` is a finite collection of three-dimensional points with shape `(N, 3)`. Empty input is allowed.

`corners` is either `None` or a finite array with shape `(M, 3)` and at least four rows. Only the first four rows define the box:

| Row | Meaning |
| --- | --- |
| `corners[0]` | reference corner $\mathbf{c}_0$ |
| `corners[1]` | endpoint of the first edge leaving $\mathbf{c}_0$ |
| `corners[2]` | endpoint of the second edge leaving $\mathbf{c}_0$ |
| `corners[3]` | endpoint of the third edge leaving $\mathbf{c}_0$ |

The three edge vectors must have positive length and be mutually perpendicular. This is the same first-four-corner convention used by `nematics3d.geometry.get_box_corners()`. Additional rows are accepted but are not needed for the calculation.

### Options

| Argument | Meaning |
| --- | --- |
| `is_return_mask=False` | Return only the selected points. |
| `is_return_mask=True` | Also return the boolean selection mask over the original rows of `points`. |
| `atol=1e-9` | Use this non-negative absolute tolerance at each of the six box faces. |

If `corners=None`, box filtering is disabled and every input point is selected.

### Returned result

With the default `is_return_mask=False`, the function returns a floating-point `NumPy` array containing the selected rows of `points`.

With `is_return_mask=True`, it returns `(selected, mask)`, where `mask` is a boolean array of shape `(N,)` referring to the original point order. This is useful when another array is indexed in parallel with `points`.


## Examples


### Return the selection mask

Set `is_return_mask=True` when the same geometric selection must also be applied to data whose rows correspond to the input points.


In [ ]:
selected, mask = n3d.geometry.select_points_in_box(
    points,
    corners,
    is_return_mask=True,
)

values = np.array([10.0, 20.0, 30.0, 40.0])
print("mask:", mask)
print("selected points:\n", selected)
print("corresponding values:", values[mask])

### Use a rotated and translated box

The box does not need to be axis-aligned. Here the first two box directions are rotated by $45^\circ$ in the $xy$ plane, while the third remains parallel to $z$.


In [ ]:
origin = np.array([3.0, -1.0, 2.0])
axis1 = np.array([1.0, 1.0, 0.0]) / np.sqrt(2.0)
axis2 = np.array([-1.0, 1.0, 0.0]) / np.sqrt(2.0)
axis3 = np.array([0.0, 0.0, 1.0])

corners_rotated = np.array([
    origin,
    origin + 2.0 * axis1,
    origin + 1.0 * axis2,
    origin + 0.5 * axis3,
])

points_rotated = np.array([
    origin + 1.0 * axis1 + 0.5 * axis2 + 0.25 * axis3,
    origin + 2.2 * axis1 + 0.5 * axis2 + 0.25 * axis3,
])

print(n3d.geometry.select_points_in_box(points_rotated, corners_rotated))

### Control boundary tolerance

`atol` is an absolute geometric tolerance. Points exactly on a face are included, and points slightly beyond a face can also be included when their excess is no larger than `atol`. Because the tolerance is absolute, interpret it in the same coordinate units as `points` and `corners`.


In [ ]:
near_boundary = np.array([
    [2.0, 0.5, 0.5],
    [2.0 + 5e-10, 0.5, 0.5],
    [2.0 + 2e-9, 0.5, 0.5],
])

_, mask = n3d.geometry.select_points_in_box(
    near_boundary,
    corners,
    is_return_mask=True,
)
print(mask)

### Disable box filtering with `corners=None`

Some workflows make clipping optional. Passing `corners=None` selects every point while preserving the same return convention.


In [ ]:
selected_all, mask_all = n3d.geometry.select_points_in_box(
    points,
    None,
    is_return_mask=True,
)
print(selected_all)
print(mask_all)

## Details

Let $\mathbf{c}_0$ be the reference corner. From the next three corners, the function constructs the edge vectors

$$\mathbf{e}_i = \mathbf{c}_i - \mathbf{c}_0, \qquad i=1,2,3,$$

their lengths $L_i=\lVert\mathbf{e}_i\rVert$, and the corresponding unit directions $\hat{\mathbf{e}}_i=\mathbf{e}_i/L_i$. The three edge vectors must be nonzero and mutually perpendicular.

For every point $\mathbf{x}$, the function projects the displacement from the reference corner onto the three box axes:

$$u_i=(\mathbf{x}-\mathbf{c}_0)\cdot\hat{\mathbf{e}}_i.$$

The point is selected when all three local coordinates satisfy

$$-\mathrm{atol} \le u_i \le L_i+\mathrm{atol}. $$

This formulation makes translation and rotation irrelevant to the membership test: the calculation is performed in the box's own orthonormal coordinate frame. A skew parallelepiped is intentionally rejected rather than silently treated as a rectangular box.

The calculation is vectorized over the input points. For $N$ three-dimensional points, the dominant work and storage therefore scale linearly with $N$.
